In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, year, month, monotonically_increasing_id

# 1. Initialize Spark Session
spark = (
    SparkSession.builder
    .appName("Build_Fact_Sales")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [2]:
# 2. Load ALL Required Data 
orders = spark.read.parquet("/user/student/cleaned_data/orders_clean")
items = spark.read.parquet("/user/student/cleaned_data/order_items_clean")
payments = spark.read.parquet("/user/student/cleaned_data/order_payments_cleaned")
customers = spark.read.parquet("/user/student/cleaned_data/customers_clean")
dim_customer = spark.read.parquet("/user/student/dim_data/dim_customer")
dim_product = spark.read.parquet("/user/student/dim_data/dim_product") # Added Dim_Product

In [3]:
# 3. Handle Multiple Payments 
primary_payments = payments.filter(col("payment_sequential") == 1)

In [4]:
# 4. Map Orders to 'cust_key'
orders_with_unique = orders.join(
    customers.select("customer_id", "customer_unique_id"), 
    on="customer_id", 
    how="inner"
)
orders_with_cust_key = orders_with_unique.join(
    dim_customer.select("customer_unique_id", "cust_key"), 
    on="customer_unique_id", 
    how="inner"
)

In [5]:
# 5. Build Base Fact DataFrame
base_fact_df = (
    items.join(orders_with_cust_key, on="order_id", how="inner")
         .join(primary_payments, on="order_id", how="left")
)

In [6]:
# 6. The SCD Type 2 Bridge: Map product_id to product_key based on date
fact_df = base_fact_df.join(
    dim_product,
    (base_fact_df.product_id == dim_product.product_id) &
    (col("order_purchase_timestamp").cast("date") >= col("dw_start_date")) &
    (col("order_purchase_timestamp").cast("date") <= col("dw_end_date")),
    how="inner"
)

In [7]:
# 7. Build the Final Fact Table
fact_sales = fact_df.select(
    col("order_id"),
    col("cust_key"),
    col("product_key"),
    col("order_status"),
    base_fact_df["price"].cast("double").alias("price"),
    col("freight_value").cast("double"),  
    col("payment_value").cast("double"),  
    col("payment_type"),
    date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast("int").alias("order_purchase_date_key"),
    year(col("order_purchase_timestamp")).alias("order_year"),
    month(col("order_purchase_timestamp")).alias("order_month")
).withColumn("sales_key", monotonically_increasing_id())

In [8]:
# 8. Reorder Columns 
final_fact_sales = fact_sales.select(
    "sales_key",
    "order_id",
    "cust_key",
    "product_key",
    "order_purchase_date_key",
    "order_status",
    "price",
    "freight_value",
    "payment_value",
    "payment_type",
    "order_year",
    "order_month"
)

final_fact_sales.show(5)

+---------+--------------------+--------+-------------+-----------------------+------------+-----+-------------+-------------+------------+----------+-----------+
|sales_key|            order_id|cust_key|  product_key|order_purchase_date_key|order_status|price|freight_value|payment_value|payment_type|order_year|order_month|
+---------+--------------------+--------+-------------+-----------------------+------------+-----+-------------+-------------+------------+----------+-----------+
|        0|000aed2e25dbad2f9...|   37571|1073741824070|               20180511|   delivered|144.0|         8.77|       152.77| credit_card|      2018|          5|
|        1|008dd5e80ebf8f849...|   24405| 274877907033|               20180320|   delivered|199.0|         8.74|       207.74|      boleto|      2018|          3|
|        2|00cf47526e0f7920b...|   33164| 111669149813|               20170209|   delivered| 59.9|        14.59|        74.49| credit_card|      2017|          2|
|        3|0114835d7f0

In [9]:
# 9. Write to HDFS
print("Writing partitioned fact table to HDFS...")
final_fact_sales.write.mode("overwrite") \
    .partitionBy("order_year", "order_month") \
    .parquet("/user/student/fact_data/fact_sales")

Writing partitioned fact table to HDFS...
